In [ ]:
print("IMPORTING MODULES")

import os, sys, subprocess, tempfile
from coffea import processor
import coffea.util
import matplotlib.pyplot as plt
import yaml

sidm_path = str(os.getcwd()).split('/sidm')[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import utilities, scaleout, sidm_processor, llpnanoaodschema
from sidm.tools.metadata import write_run_metadata
utilities.set_plot_style()
%matplotlib inline

print("VOMSPROXY")
os.environ["X509_USER_PROXY"] = "/uscms_data/d3/murtazas/x509_proxy.pem"
scaleout.check_voms_proxy()

print("CREATE CLUSTER CLIENR")
cluster, client = scaleout.make_lpc_client(
    min_workers=10,
    max_workers=100,
    memory='4GB',
    disk='4GB',
    scheduler_options={"dashboard_address": ":8791"}
)
print('dashboard:', cluster.dashboard_link)
client.wait_for_workers(1, timeout=600)
print('first worker connected; cluster:', cluster)


print("CREATE RUNNER")
runner = processor.Runner(
    executor=processor.DaskExecutor(client=client, status=False),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=True,
    chunksize=10000,
)

print("CHANNEL NAMES AND HIST COLLECTION")
# Run the slide "Before"/"After" channels and the dimuon-vertex-fit mirrors
# from this branch in one pass, so the Spread-based and vtx_chi2-based cosmic
# vetoes can be compared on identical events.
channel_names = [
    "cosmic_muons",  # the spread-inverted cosmic region (inverted spread cuts + inverted cosAlpha):
                     # no vertex information in its definition, so vtx_chi2 is evaluated
                     # where cosmics live without being used to select them
]
run_tag = "vtxCosmic_2018C"  # data-only cosmic-tail check


print("Processor creation")
hist_collections = ["cosmic_veto"]

p = sidm_processor.SidmProcessor(
    channel_names,
    hist_collections,
    # unweighted_hist=True,
)

max_files_bg = -1
max_files_data = -1
max_files_signal = -1




In [ ]:
data = [ 
'DoubleMuon_2018C',
# 'DoubleMuon_2018A_0',
# 'DoubleMuon_2018A_1',
# 'DoubleMuon_2018A_2',
# 'DoubleMuon_2018B_0',
# 'DoubleMuon_2018B_1',
# 'DoubleMuon_2018B_2',
# 'DoubleMuon_2018B_3',
# 'DoubleMuon_2018B_4',
# 'DoubleMuon_2018B_5',
# 'DoubleMuon_2018B_6',
# 'DoubleMuon_2018B_7',
# 'DoubleMuon_2018B_8',
# 'DoubleMuon_2018B_9',
# 'DoubleMuon_2018B_10',
# 'DoubleMuon_2018B_11',
# 'DoubleMuon_2018B_12',
# 'DoubleMuon_2018B_13',
# 'DoubleMuon_2018D_0',
# 'DoubleMuon_2018D_1',
# 'DoubleMuon_2018D_10',
# 'DoubleMuon_2018D_11',
# 'DoubleMuon_2018D_12',
# 'DoubleMuon_2018D_13',
# 'DoubleMuon_2018D_14',
# 'DoubleMuon_2018D_15',
# 'DoubleMuon_2018D_16',
# 'DoubleMuon_2018D_17',
# 'DoubleMuon_2018D_18',
# 'DoubleMuon_2018D_19',
# 'DoubleMuon_2018D_2',
# 'DoubleMuon_2018D_20',
# 'DoubleMuon_2018D_21',
# 'DoubleMuon_2018D_22',
# 'DoubleMuon_2018D_23',
# 'DoubleMuon_2018D_24',
# 'DoubleMuon_2018D_25',
# 'DoubleMuon_2018D_26',
# 'DoubleMuon_2018D_27',
# 'DoubleMuon_2018D_28',
# 'DoubleMuon_2018D_29',
# 'DoubleMuon_2018D_3',
# 'DoubleMuon_2018D_30',
# 'DoubleMuon_2018D_31',
# 'DoubleMuon_2018D_32',
# 'DoubleMuon_2018D_33',
# 'DoubleMuon_2018D_34',
# 'DoubleMuon_2018D_35',
# 'DoubleMuon_2018D_36',
# 'DoubleMuon_2018D_37',
# 'DoubleMuon_2018D_38',
# 'DoubleMuon_2018D_39',
# 'DoubleMuon_2018D_4',
# 'DoubleMuon_2018D_40',
# 'DoubleMuon_2018D_41',
# 'DoubleMuon_2018D_42',
# 'DoubleMuon_2018D_43',
# 'DoubleMuon_2018D_44',
# 'DoubleMuon_2018D_45',
# 'DoubleMuon_2018D_46',
# 'DoubleMuon_2018D_47',
# 'DoubleMuon_2018D_48',
# 'DoubleMuon_2018D_49',
# 'DoubleMuon_2018D_5',
# 'DoubleMuon_2018D_6',
# 'DoubleMuon_2018D_7',
# 'DoubleMuon_2018D_8',
# 'DoubleMuon_2018D_9',
]
print("Processing DATA")
for x in data:
    print(x)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"
    if os.path.exists(f"{dir_path}/{x}.coffea"):
        print("output exists, skipping")
        continue
    fileset = utilities.make_fileset([x],'llpNanoAOD_v2',location_cfg='data_skimmed.yaml',
                                     max_files=max_files_data,replace_xcache=True,)
    output_data = runner.run(fileset, treename='Events', processor_instance=p)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"  # outside sidm/ so UploadDirectory does not ship outputs to workers
    os.makedirs(dir_path, exist_ok=True)
    coffea.util.save(output_data, f"{dir_path}/{x}.coffea"  )
    # Provenance sidecar: selections, hist collections, input file list, per-sample
    # cross section, SIDM commit and coffea version, next to the .coffea it describes.
    meta_local = write_run_metadata(
        f"{dir_path}/{x}.coffea",
        fileset=fileset,
        selections=channel_names,
        hist_collections=hist_collections,
        schema="LLPNanoAODSchema",
        chunksize=10000,
        extra={"run_tag": run_tag},
    )
    REDIR   = "root://cmseos.fnal.gov"
    EOS_DIR = f"/store/group/lpcmetx/SIDM/coffea_outputs/{os.environ['USER']}/{run_tag}"
    subprocess.run(["xrdfs", REDIR, "mkdir", "-p", EOS_DIR], check=True)
    with tempfile.TemporaryDirectory() as tmp:
        coffea_local = os.path.join(tmp, f"{x}.coffea")
        coffea.util.save(output_data, coffea_local)
        print(f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}")
        subprocess.run(["xrdcp", "-f", coffea_local, f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}"], 
                       check=True)
    subprocess.run(["xrdcp", "-f", meta_local,
                    f"{REDIR}/{EOS_DIR}/{os.path.basename(meta_local)}"], check=True)
    
    
   




In [ ]:



bgs = []  # data-only study

for x in bgs:
    print(x)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"
    if os.path.exists(f"{dir_path}/{x}.coffea"):
        print("output exists, skipping")
        continue
    fileset = utilities.make_fileset([x], "skimmed_llpNanoAOD_v2", location_cfg="backgrounds.yaml",
                                     max_files=max_files_bg,
                                     replace_xcache=True, )
    print(len(fileset[x]["files"]))
    output = runner.run(fileset, treename="Events", processor_instance=p)
    dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"  # outside sidm/ so UploadDirectory does not ship outputs to workers
    os.makedirs(dir_path, exist_ok=True)
    coffea.util.save(output, f"{dir_path}/{x}.coffea"  )
    # Provenance sidecar: selections, hist collections, input file list, per-sample
    # cross section, SIDM commit and coffea version, next to the .coffea it describes.
    meta_local = write_run_metadata(
        f"{dir_path}/{x}.coffea",
        fileset=fileset,
        selections=channel_names,
        hist_collections=hist_collections,
        schema="LLPNanoAODSchema",
        chunksize=10000,
        extra={"run_tag": run_tag},
    )
    REDIR   = "root://cmseos.fnal.gov"
    EOS_DIR = f"/store/group/lpcmetx/SIDM/coffea_outputs/{os.environ['USER']}/{run_tag}"
    subprocess.run(["xrdfs", REDIR, "mkdir", "-p", EOS_DIR], check=True)
    with tempfile.TemporaryDirectory() as tmp:
        coffea_local = os.path.join(tmp, f"{x}.coffea")
        coffea.util.save(output, coffea_local)
        print(f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}")
        subprocess.run(["xrdcp", "-f", coffea_local, f"{REDIR}/{EOS_DIR}/{os.path.basename(coffea_local)}"],
                       check=True)
    subprocess.run(["xrdcp", "-f", meta_local,
                    f"{REDIR}/{EOS_DIR}/{os.path.basename(meta_local)}"], check=True)

In [4]:
client.close()

cluster.close()